In [1]:
# %% Cell 1: Setup and Imports
import numpy as np
import pandas as pd
import nibabel as nib
import bct  # Brain Connectivity Toolbox
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn import plotting
from nilearn.connectome import ConnectivityMeasure
import networkx as nx
from glob import glob
from pathlib import Path
import os
import re
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")



In [2]:
# %% Cell 2: Configuration for XCP-D Output
# === CONFIGURATION ===
XCP_OUTPUT_DIR = '/projects/aabdulrasul/TAY/GT/data'
OUTPUT_DIR = '/projects/aabdulrasul/TAY/GT/test'

# === PARCELLATION SELECTION ===
PARCELLATIONS = ['4S456Parcels', 'Glasser', 'Gordon']

# === EXPECTED PARCEL COUNTS (for robust orientation check) ===
EXPECTED_PARCELS = {
    '4S156Parcels': 156, '4S256Parcels': 256, '4S356Parcels': 356,
    '4S456Parcels': 456, '4S556Parcels': 556, '4S656Parcels': 656,
    '4S756Parcels': 756, '4S856Parcels': 856, '4S956Parcels': 956,
    '4S1056Parcels': 1056, 'Glasser': 360, 'Gordon': 333,
    'HCP': 360, 'MIDB': 82, 'MyersLabonte': 246, 'Tian': 54
}

# === PROCESSING OPTIONS ===
USE_CONCATENATED = False

# === EDGE HANDLING (CRITICAL CHOICE) ===
# 'positive_only' - Keep only positive correlations (RECOMMENDED for standard graph theory)
# 'absolute' - Use absolute value (unsigned network - must justify)
# 'signed' - Keep both positive and negative (requires specialized analysis)
EDGE_TYPE = 'positive_only'


## NP seed -- for reproducibility for the Louvain algorithm and random networks
np.random.seed(23)  

# === GRAPH THEORY PARAMETERS ===
# Thresholds for density sweep (recommended for robustness)
THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
PRIMARY_THRESHOLD = 0.15  # Primary threshold for detailed analysis

THRESHOLD_TYPE = 'proportional'
COMPUTE_SMALLWORLD = True
N_RANDOM = 100  # Number of random networks for small-worldness

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'metrics'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'connectivity_matrices'), exist_ok=True)

print(f"XCP-D Output Directory: {XCP_OUTPUT_DIR}")
print(f"Results Output Directory: {OUTPUT_DIR}")
print(f"Edge handling: {EDGE_TYPE}")
print(f"Parcellations: {PARCELLATIONS}")
print(f"Threshold sweep: {THRESHOLDS}")
print(f"Primary threshold: {PRIMARY_THRESHOLD}")
print(f"Random seed: 23")


XCP-D Output Directory: /projects/aabdulrasul/TAY/GT/data
Results Output Directory: /projects/aabdulrasul/TAY/GT/test
Edge handling: positive_only
Parcellations: ['4S456Parcels', 'Glasser', 'Gordon']
Threshold sweep: [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
Primary threshold: 0.15
Random seed: 23


In [3]:
# %% Cell 3: XCP-D File Discovery Functions
def find_xcp_subjects(xcp_dir):
    """Find all subjects in XCP-D output directory."""
    xcp_path = Path(xcp_dir)
    subjects = sorted([d.name for d in xcp_path.glob('sub-*') if d.is_dir()])
    return subjects

def find_sessions(subject_dir):
    """Find all sessions for a subject."""
    return sorted([d.name for d in Path(subject_dir).glob('ses-*') if d.is_dir()])

def find_ptseries_files(func_dir, parcellation=None, concatenated=True, task='rest'):
    """
    Find ptseries files in a func directory.
    """
    func_path = Path(func_dir)
    
    pattern = '*_stat-mean_timeseries.ptseries.nii'
    files = list(func_path.glob(pattern))
    
    if task:
        files = [f for f in files if f'task-{task}' in f.name]
    
    if concatenated:
        files = [f for f in files if '_run-' not in f.name]
    else:
        files = [f for f in files if '_run-' in f.name]
    
    if parcellation:
        files = [f for f in files if f'seg-{parcellation}' in f.name]
    
    files = sorted(files)
    
    # Only warn for concatenated mode where multiple files is unexpected
    if concatenated and len(files) > 1:
        print(f"  Warning: Multiple concatenated files found for {parcellation}")
    
    return files

def get_available_parcellations(func_dir, concatenated=True):
    """Get list of all available parcellations in a func directory."""
    files = find_ptseries_files(func_dir, parcellation=None, concatenated=concatenated)
    parcellations = set()
    
    for f in files:
        match = re.search(r'seg-([^_]+)_', f.name)
        if match:
            parcellations.add(match.group(1))
    
    return sorted(parcellations)

def load_ptseries(filepath, parcellation=None, expected_parcels=None):
    """
    Load parcellated time series from XCP-D ptseries file.
    
    CIFTI ptseries convention: (timepoints, parcels)
    
    Parameters:
    -----------
    filepath : str or Path
        Path to ptseries file
    parcellation : str, optional
        Parcellation name for lookup in expected_parcels
    expected_parcels : dict, optional
        Dictionary mapping parcellation names to expected parcel counts
    """
    img = nib.load(filepath)
    data = img.get_fdata()
    
    if data.ndim == 1:
        raise ValueError(f"1D data in {filepath}")
    
    # Robust orientation check
    if expected_parcels and parcellation and parcellation in expected_parcels:
        n_expected = expected_parcels[parcellation]
        if data.shape[1] == n_expected:
            pass  # Correct orientation: (timepoints, parcels)
        elif data.shape[0] == n_expected:
            print(f"  Note: Transposing data from shape {data.shape} (expected {n_expected} parcels)")
            data = data.T
        else:
            print(f"  Warning: Neither dimension matches expected {n_expected} parcels for {parcellation}")
            # Fall back to heuristic
            if data.shape[0] < data.shape[1]:
                data = data.T
    else:
        # Heuristic fallback: assume more timepoints than parcels
        if data.shape[0] < data.shape[1]:
            print(f"  Note: Transposing data from shape {data.shape}")
            data = data.T
    
    return data

In [4]:
# %% Cell 4: Connectivity Matrix Functions (CORRECTED)
def compute_connectivity_matrix(timeseries, method='pearson'):
    """
    Compute functional connectivity matrix from parcellated time series.
    Returns correlation coefficients (r values), NOT Fisher z.
    """
    if method == 'pearson':
        conn_matrix = np.corrcoef(timeseries.T)
    elif method == 'partial':
        measure = ConnectivityMeasure(kind='partial correlation')
        conn_matrix = measure.fit_transform([timeseries])[0]
    
    # Clean up: remove diagonal, handle NaN, ensure symmetry
    np.fill_diagonal(conn_matrix, 0)
    conn_matrix = np.nan_to_num(conn_matrix)
    conn_matrix = (conn_matrix + conn_matrix.T) / 2  # Enforce symmetry
    
    return conn_matrix

def prepare_graph_matrix(conn_matrix, edge_type='positive_only'):
    """
    Prepare connectivity matrix for graph analysis based on edge handling choice.
    """
    if edge_type == 'positive_only':
        graph_matrix = np.clip(conn_matrix, 0, None)
    elif edge_type == 'absolute':
        graph_matrix = np.abs(conn_matrix)
    elif edge_type == 'signed':
        graph_matrix = conn_matrix.copy()
    else:
        raise ValueError(f"Unknown edge_type: {edge_type}")
    
    # Ensure symmetry and zero diagonal
    graph_matrix = (graph_matrix + graph_matrix.T) / 2
    np.fill_diagonal(graph_matrix, 0)
    
    return graph_matrix

def threshold_matrix(conn_matrix, threshold, threshold_type='proportional', binarize=False):
    """
    Threshold connectivity matrix.
    
    IMPORTANT: Works on r-values (or prepared graph matrix), NOT Fisher-z.
    """
    if threshold_type == 'proportional':
        thresholded = bct.threshold_proportional(conn_matrix, threshold)
    elif threshold_type == 'absolute':
        thresholded = conn_matrix.copy()
        thresholded[thresholded < threshold] = 0
    else:
        raise ValueError(f"Unknown threshold_type: {threshold_type}")
    
    # Ensure symmetry and zero diagonal after thresholding
    thresholded = (thresholded + thresholded.T) / 2
    np.fill_diagonal(thresholded, 0)
    
    if binarize:
        thresholded = (thresholded > 0).astype(float)
    
    return thresholded

def compute_density(A):
    """
    Compute density of an undirected graph correctly.
    
    For undirected graphs: density = m / (N*(N-1)/2)
    where m = number of edges (upper triangle only)
    """
    np.fill_diagonal(A, 0)
    m = np.sum(np.triu(A, 1) > 0)  # Count edges in upper triangle only
    n = A.shape[0]
    max_edges = n * (n - 1) / 2
    return m / max_edges if max_edges > 0 else 0

In [5]:
# %% Cell 5: Graph Theory Metrics (CORRECTED)
def compute_weighted_metrics(W):
    """
    Compute graph metrics on WEIGHTED network.
    
    Parameters:
    -----------
    W : np.ndarray
        Weighted adjacency matrix (non-negative, thresholded)
        
    Returns:
    --------
    metrics : dict
        Dictionary of weighted graph metrics
    """
    metrics = {}
    
    # Ensure non-negative weights, symmetry, zero diagonal
    W = np.clip(W, 0, None)
    W = (W + W.T) / 2
    np.fill_diagonal(W, 0)
    
    # 1. Global Efficiency (weighted)
    metrics['global_efficiency'] = bct.efficiency_wei(W)
    
    # 2. Clustering Coefficient (weighted)
    clustering = bct.clustering_coef_wu(W)
    metrics['clustering_nodal'] = clustering
    metrics['clustering_mean'] = np.nanmean(clustering)
    
    # 3. Characteristic Path Length using BCT standard conversion
    # Use BCT's weight_conversion for consistency with literature
    D = bct.weight_conversion(W, 'lengths')
    D[W == 0] = np.inf
    np.fill_diagonal(D, 0)
    
    try:
        charpath, efficiency, ecc, radius, diameter = bct.charpath(D, include_infinite=False)
        metrics['characteristic_path_length'] = charpath if np.isfinite(charpath) else np.nan
        metrics['radius'] = radius if np.isfinite(radius) else np.nan
        metrics['diameter'] = diameter if np.isfinite(diameter) else np.nan
    except:
        charpath, efficiency, ecc, radius, diameter = bct.charpath(D)
        metrics['characteristic_path_length'] = charpath if np.isfinite(charpath) else np.nan
        metrics['radius'] = radius if np.isfinite(radius) else np.nan
        metrics['diameter'] = diameter if np.isfinite(diameter) else np.nan
    
    # 4. Modularity - FIXED: best of 10 runs for reproducibility
    best_q = -np.inf
    best_ci = None
    for _ in range(10):
        ci_temp, q_temp = bct.community_louvain(W)
        if q_temp > best_q:
            best_q = q_temp
            best_ci = ci_temp
    ci, q = best_ci, best_q
    metrics['modularity'] = q
    metrics['community_assignments'] = ci
    metrics['n_modules'] = len(np.unique(ci))

    # 5. Node Strength
    strength = bct.strengths_und(W)
    metrics['strength'] = strength
    metrics['strength_mean'] = np.mean(strength)
    
    # 6. Betweenness Centrality (weighted)
    try:
        bc = bct.betweenness_wei(D)
        bc = np.where(np.isinf(bc), 0, bc)
    except:
        bc = np.zeros(W.shape[0])
    metrics['betweenness_centrality'] = bc
    metrics['betweenness_mean'] = np.nanmean(bc)
    
    # 7. Participation Coefficient
    pc = bct.participation_coef(W, ci)
    metrics['participation_coefficient'] = pc
    metrics['participation_mean'] = np.nanmean(pc)
    
    # 8. Local Efficiency (weighted)
    local_eff = bct.efficiency_wei(W, local=True)
    metrics['local_efficiency'] = local_eff
    metrics['local_efficiency_mean'] = np.nanmean(local_eff)
    
    # 9. Assortativity (weighted)
    try:
        metrics['assortativity'] = bct.assortativity_wei(W, flag=0)
    except:
        metrics['assortativity'] = np.nan
    
    # 10. Transitivity (weighted)
    metrics['transitivity'] = bct.transitivity_wu(W)
    
    return metrics

def compute_binary_metrics(A):
    """
    Compute graph metrics on BINARY network.
    """
    metrics = {}
    
    # Ensure binary, symmetric, zero diagonal
    A = (A > 0).astype(float)
    A = (A + A.T) / 2
    A = (A > 0).astype(float)
    np.fill_diagonal(A, 0)
    
    # 1. Global Efficiency
    metrics['global_efficiency'] = bct.efficiency_bin(A)
    
    # 2. Clustering Coefficient
    clustering = bct.clustering_coef_bu(A)
    metrics['clustering_nodal'] = clustering
    metrics['clustering_mean'] = np.nanmean(clustering)
    
    # 3. Characteristic Path Length
    D = bct.distance_bin(A)
    try:
        charpath, efficiency, ecc, radius, diameter = bct.charpath(D, include_infinite=False)
    except:
        charpath, efficiency, ecc, radius, diameter = bct.charpath(D)
    
    metrics['characteristic_path_length'] = charpath if np.isfinite(charpath) else np.nan
    metrics['radius'] = radius if np.isfinite(radius) else np.nan
    metrics['diameter'] = diameter if np.isfinite(diameter) else np.nan
    
    # 4. Modularity - FIXED: Use A, not W; best of 10 runs
    best_q = -np.inf
    best_ci = None
    for _ in range(10):
        ci_temp, q_temp = bct.community_louvain(A)  # FIXED: was W, now A
        if q_temp > best_q:
            best_q = q_temp
            best_ci = ci_temp
    ci, q = best_ci, best_q
    metrics['modularity'] = q
    metrics['community_assignments'] = ci
    metrics['n_modules'] = len(np.unique(ci))
    
    # 5. Degree
    degree = bct.degrees_und(A)
    metrics['degree'] = degree
    metrics['degree_mean'] = np.mean(degree)
    
    # 6. Betweenness Centrality
    bc = bct.betweenness_bin(A)
    metrics['betweenness_centrality'] = bc
    metrics['betweenness_mean'] = np.nanmean(bc)
    
    # 7. Participation Coefficient
    pc = bct.participation_coef(A, ci)
    metrics['participation_coefficient'] = pc
    metrics['participation_mean'] = np.nanmean(pc)
    
    # 8. Local Efficiency
    local_eff = bct.efficiency_bin(A, local=True)
    metrics['local_efficiency'] = local_eff
    metrics['local_efficiency_mean'] = np.nanmean(local_eff)
    
    # 9. Assortativity
    try:
        metrics['assortativity'] = bct.assortativity_bin(A, flag=0)
    except:
        metrics['assortativity'] = np.nan
    
    # 10. Transitivity
    metrics['transitivity'] = bct.transitivity_bu(A)
    
    # 11. Rich Club (only valid for binary)
    try:
        rc = bct.rich_club_bu(A)
        metrics['rich_club'] = rc
    except:
        metrics['rich_club'] = np.array([])
    
    return metrics

def compute_small_worldness_binary(A, n_random=100):
    """
    Compute small-worldness on BINARY network using proper null model.
    
    This is the CORRECT approach: randmio_und is valid for binary networks.
    """
    # Ensure binary, symmetric, zero diagonal
    A = (A > 0).astype(float)
    A = (A + A.T) / 2
    A = (A > 0).astype(float)
    np.fill_diagonal(A, 0)
    
    # Real network metrics
    C_real = np.mean(bct.clustering_coef_bu(A))
    D_real = bct.distance_bin(A)
    
    try:
        L_real, _, _, _, _ = bct.charpath(D_real, include_infinite=False)
    except:
        L_real, _, _, _, _ = bct.charpath(D_real)
    
    if not np.isfinite(L_real):
        print("  Warning: Network disconnected, cannot compute small-worldness")
        return {
            'sigma': np.nan, 'gamma': np.nan, 'lambda': np.nan,
            'C_real': C_real, 'C_rand': np.nan, 'L_real': np.nan, 'L_rand': np.nan
        }
    
    # Random networks (degree-preserving rewiring)
    C_rand_list = []
    L_rand_list = []
    
    for _ in range(n_random):
        try:
            rand_net, _ = bct.randmio_und(A, 10)
            
            C_rand = np.mean(bct.clustering_coef_bu(rand_net))
            D_rand = bct.distance_bin(rand_net)
            
            try:
                L_rand, _, _, _, _ = bct.charpath(D_rand, include_infinite=False)
            except:
                L_rand, _, _, _, _ = bct.charpath(D_rand)
            
            if np.isfinite(L_rand):
                C_rand_list.append(C_rand)
                L_rand_list.append(L_rand)
        except Exception:
            continue
    
    if len(C_rand_list) < 10:
        print(f"  Warning: Only {len(C_rand_list)} valid random networks generated")
        if len(C_rand_list) == 0:
            return {
                'sigma': np.nan, 'gamma': np.nan, 'lambda': np.nan,
                'C_real': C_real, 'C_rand': np.nan, 'L_real': L_real, 'L_rand': np.nan
            }
    
    C_rand_mean = np.mean(C_rand_list)
    L_rand_mean = np.mean(L_rand_list)
    
    # Safety check for division
    if C_rand_mean > 0 and L_rand_mean > 0:
        gamma = C_real / C_rand_mean
        lam = L_real / L_rand_mean
        sigma = gamma / lam
    else:
        gamma = np.nan
        lam = np.nan
        sigma = np.nan
    
    return {
        'sigma': sigma, 'gamma': gamma, 'lambda': lam,
        'C_real': C_real, 'C_rand': C_rand_mean, 'L_real': L_real, 'L_rand': L_rand_mean
    }

In [6]:

# %% Cell 6: Visualization Functions
def plot_connectivity_matrix(conn_matrix, title="Connectivity Matrix", save_path=None):
    """Plot connectivity matrix as heatmap."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    vmax = np.percentile(np.abs(conn_matrix), 95)
    im = ax.imshow(conn_matrix, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='equal')
    
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Correlation (r)', fontsize=11)
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Parcels', fontsize=11)
    ax.set_ylabel('Parcels', fontsize=11)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    return fig

def plot_community_matrix(conn_matrix, community_labels, title="Community Structure", save_path=None):
    """Plot connectivity matrix reordered by community."""
    sorted_idx = np.argsort(community_labels)
    sorted_matrix = conn_matrix[np.ix_(sorted_idx, sorted_idx)]
    sorted_communities = community_labels[sorted_idx]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    vmax = np.percentile(np.abs(sorted_matrix), 95)
    im = ax.imshow(sorted_matrix, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='equal')
    
    # Add community boundaries
    unique_communities = np.unique(sorted_communities)
    for comm in unique_communities:
        idx = np.where(sorted_communities == comm)[0]
        if len(idx) > 0:
            boundary = idx[-1] + 0.5
            ax.axhline(boundary, color='black', linewidth=1.5)
            ax.axvline(boundary, color='black', linewidth=1.5)
    
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Correlation (r)', fontsize=11)
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Parcels (sorted by community)', fontsize=11)
    ax.set_ylabel('Parcels (sorted by community)', fontsize=11)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    return fig

def plot_metrics_summary(metrics_df, parcellation, save_path=None):
    """Create summary visualization of metrics distribution."""
    metrics_to_plot = ['global_efficiency_w', 'clustering_mean_w', 'modularity_w',
                      'characteristic_path_length_w', 'transitivity_w', 'small_worldness_sigma']
    
    # Fallback to available columns
    available_metrics = [m for m in metrics_to_plot if m in metrics_df.columns]
    
    if len(available_metrics) == 0:
        # Try without suffix
        metrics_to_plot = ['global_efficiency', 'clustering_mean', 'modularity',
                          'characteristic_path_length', 'transitivity', 'small_worldness_sigma']
        available_metrics = [m for m in metrics_to_plot if m in metrics_df.columns]
    
    n_metrics = len(available_metrics)
    
    if n_metrics == 0:
        print("No metrics to plot")
        return None
    
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    
    for i, metric in enumerate(available_metrics):
        ax = axes[i]
        # Remove inf and nan values
        data = metrics_df[metric].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(data) == 0:
            ax.text(0.5, 0.5, f'No valid data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(metric.replace('_', ' ').title(), fontsize=10)
            continue
        
        ax.hist(data, bins=min(20, max(5, len(data)//2)), alpha=0.7, color='steelblue', edgecolor='black')
        ax.axvline(np.mean(data), color='red', linestyle='--', linewidth=2)
        ax.set_xlabel(metric.replace('_', ' ').title(), fontsize=10)
        ax.set_ylabel('Count', fontsize=10)
        ax.set_title(f'μ = {np.mean(data):.3f} ± {np.std(data):.3f}', fontsize=10)
    
    for j in range(len(available_metrics), len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle(f'Graph Metrics Distribution - {parcellation}\n(n = {len(metrics_df)} subjects)', 
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    return fig

def plot_threshold_sweep(sweep_df, subject_id, save_path=None):
    """Plot metrics across different thresholds (AUC analysis)."""
    metrics_to_plot = ['global_efficiency_w', 'clustering_mean_w', 'modularity_w',
                      'characteristic_path_length_w', 'transitivity_w', 'small_worldness_sigma']
    
    available = [m for m in metrics_to_plot if m in sweep_df.columns]
    
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    
    for i, metric in enumerate(available):
        ax = axes[i]
        data = sweep_df[['threshold', metric]].dropna()
        
        if len(data) > 0:
            ax.plot(data['threshold'], data[metric], 'o-', linewidth=2, markersize=8, color='steelblue')
            
            # Compute AUC
            if len(data) > 1:
                auc = np.trapz(data[metric], data['threshold'])
                ax.set_title(f'{metric.replace("_", " ").title()}\nAUC = {auc:.4f}', fontsize=10)
            else:
                ax.set_title(metric.replace('_', ' ').title(), fontsize=10)
        
        ax.set_xlabel('Threshold (density)', fontsize=10)
        ax.set_ylabel('Value', fontsize=10)
        ax.grid(True, alpha=0.3)
    
    for j in range(len(available), len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle(f'Threshold Sweep - {subject_id}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    return fig

def plot_strength_distribution(strength_values, title="Strength Distribution", save_path=None):
    """Plot node strength distribution."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(strength_values, bins=30, density=True, alpha=0.7, 
                 color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Node Strength', fontsize=11)
    axes[0].set_ylabel('Density', fontsize=11)
    axes[0].set_title('Histogram', fontsize=11, fontweight='bold')
    
    hist, bins = np.histogram(strength_values, bins=30)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    mask = hist > 0
    if np.any(mask):
        axes[1].loglog(bin_centers[mask], hist[mask], 'o', markersize=6, color='steelblue')
    axes[1].set_xlabel('Node Strength (log)', fontsize=11)
    axes[1].set_ylabel('Frequency (log)', fontsize=11)
    axes[1].set_title('Log-Log Plot', fontsize=11, fontweight='bold')
    
    plt.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    return fig

def plot_network_graph(conn_matrix, community_labels=None, node_metric=None,
                       title="Network Graph", save_path=None):
    """Create network visualization using NetworkX."""
    conn_vis = conn_matrix.copy()
    threshold = np.percentile(conn_vis[conn_vis > 0], 70) if np.any(conn_vis > 0) else 0
    conn_vis[conn_vis < threshold] = 0
    
    G = nx.from_numpy_array(conn_vis)
    G.remove_nodes_from(list(nx.isolates(G)))
    
    if len(G.nodes()) == 0:
        print("No nodes to display after thresholding")
        return None
    
    fig, ax = plt.subplots(figsize=(12, 12))
    
    if node_metric is not None:
        node_metric = np.nan_to_num(node_metric, nan=0)
        sizes = 50 + 300 * (node_metric - np.min(node_metric)) / \
                (np.max(node_metric) - np.min(node_metric) + 1e-10)
        sizes = [sizes[n] if n < len(sizes) else 100 for n in G.nodes()]
    else:
        sizes = 100
    
    if community_labels is not None:
        colors = [community_labels[n] if n < len(community_labels) else 0 for n in G.nodes()]
        cmap = plt.cm.tab20
    else:
        colors = 'steelblue'
        cmap = None
    
    pos = nx.spring_layout(G, k=2/np.sqrt(len(G.nodes())), iterations=50, seed=42)
    
    edges = G.edges()
    weights = [G[u][v]['weight'] for u, v in edges]
    
    nx.draw_networkx_edges(G, pos, alpha=0.2, width=np.array(weights)*2, 
                          edge_color='gray', ax=ax)
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors,
                          cmap=cmap, alpha=0.8, ax=ax)
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    return fig



In [12]:
# %% Cell 7: Main XCP-D Graph Theory Pipeline (CORRECTED)
class XCPGraphTheoryPipeline:
    """
    Graph theory analysis pipeline for XCP-D outputs.
    
    CORRECTED VERSION v3:
    - Fixed to process ALL runs (not just first)
    - Fixed density calculation
    - Using bct.weight_conversion for path lengths
    - Deterministic primary threshold selection
    - Robust ptseries orientation check
    - Enforced symmetry throughout
    """
    
    def __init__(self, xcp_dir, output_dir, parcellations=None, 
                 use_concatenated=False, thresholds=[0.15], threshold_type='proportional',
                 edge_type='positive_only', expected_parcels=None):
        """
        Initialize the pipeline.
        
        Parameters:
        -----------
        use_concatenated : bool
            If False (default), process each run separately.
            If True, use concatenated session-level files (not recommended).
        """
        self.xcp_dir = Path(xcp_dir)
        self.output_dir = Path(output_dir)
        self.parcellations = parcellations
        self.use_concatenated = use_concatenated
        self.thresholds = thresholds
        self.threshold_type = threshold_type
        self.edge_type = edge_type
        self.expected_parcels = expected_parcels or EXPECTED_PARCELS
        
        # Determine primary threshold ONCE (deterministic)
        self.primary_threshold = (PRIMARY_THRESHOLD if PRIMARY_THRESHOLD in self.thresholds 
                                  else self.thresholds[len(self.thresholds)//2])
        
        # Create output directories
        self.output_dir.mkdir(parents=True, exist_ok=True)
        (self.output_dir / 'figures').mkdir(exist_ok=True)
        (self.output_dir / 'metrics').mkdir(exist_ok=True)
        (self.output_dir / 'connectivity_matrices').mkdir(exist_ok=True)
        
        # Results storage
        self.results = {}
        self.summary_df = None
        
    def discover_data(self):
        """Discover all available subjects and parcellations."""
        print("Discovering XCP-D data...")
        print(f"Using per-run files: {not self.use_concatenated}")
        
        subjects = find_xcp_subjects(self.xcp_dir)
        print(f"Found {len(subjects)} subjects")
        
        available_parcellations = set()
        for subj in subjects[:5]:
            for ses_dir in (self.xcp_dir / subj).glob('ses-*'):
                func_dir = ses_dir / 'func'
                if func_dir.exists():
                    parcs = get_available_parcellations(func_dir, self.use_concatenated)
                    available_parcellations.update(parcs)
        
        print(f"Available parcellations: {sorted(available_parcellations)}")
        
        if self.parcellations is None:
            self.parcellations = sorted(available_parcellations)
        else:
            missing = set(self.parcellations) - available_parcellations
            if missing:
                print(f"Warning: Parcellations not found: {missing}")
            self.parcellations = [p for p in self.parcellations if p in available_parcellations]
        
        print(f"Will process: {self.parcellations}")
        print(f"Edge handling: {self.edge_type}")
        print(f"Thresholds: {self.thresholds}")
        print(f"Primary threshold: {self.primary_threshold}")
        
        return subjects
    
    def _extract_run_id(self, filename):
        """Extract run ID from filename (e.g., 'run-01' -> '01')."""
        match = re.search(r'run-(\d+)', str(filename))
        return match.group(1) if match else None
    
    def process_subject_session(self, ptseries_file, subject_id, session_id, 
                                 parcellation, run_id=None, compute_smallworld=True, n_random=100):
        """
        Process a single ptseries file.
        
        Parameters:
        -----------
        run_id : str, optional
            Run identifier (e.g., '01', '02'). None for concatenated files.
        """
        # Load time series with robust orientation check
        timeseries = load_ptseries(ptseries_file, parcellation=parcellation, 
                                   expected_parcels=self.expected_parcels)
        n_timepoints, n_parcels = timeseries.shape
        
        # Validate shape
        if parcellation in self.expected_parcels:
            expected = self.expected_parcels[parcellation]
            if n_parcels != expected:
                print(f"  WARNING: Expected {expected} parcels for {parcellation}, got {n_parcels}")
                print(f"  File: {ptseries_file}")
                return None
        
        if n_timepoints < 50:
            print(f"  Skipping: only {n_timepoints} timepoints")
            return None
        
        # Compute raw correlation matrix (r-values)
        conn_matrix_raw = compute_connectivity_matrix(timeseries, method='pearson')
        
        # Prepare graph matrix based on edge handling choice
        conn_matrix = prepare_graph_matrix(conn_matrix_raw, edge_type=self.edge_type)
        
        # === THRESHOLD SWEEP ===
        sweep_results = []
        primary_result = None
        
        for thresh in self.thresholds:
            # Threshold to get weighted matrix
            W = threshold_matrix(conn_matrix, thresh, threshold_type=self.threshold_type, binarize=False)
            
            # Ensure clean matrix for binary conversion
            np.fill_diagonal(W, 0)
            W = (W + W.T) / 2
            
            # Binarize for binary metrics and small-worldness
            A = (W > 0).astype(float)
            np.fill_diagonal(A, 0)
            
            # Compute CORRECT density
            density = compute_density(A)
            
            # Compute weighted metrics
            metrics_w = compute_weighted_metrics(W)
            
            # Compute binary metrics
            metrics_b = compute_binary_metrics(A)
            
            # Compute small-worldness on BINARY network
            sw_metrics = None
            if compute_smallworld:
                sw_metrics = compute_small_worldness_binary(A, n_random=n_random)
            
            # Store sweep result
            sweep_row = {
                'threshold': thresh,
                'density': density,
                # Weighted metrics
                'global_efficiency_w': metrics_w['global_efficiency'],
                'clustering_mean_w': metrics_w['clustering_mean'],
                'modularity_w': metrics_w['modularity'],
                'n_modules_w': metrics_w['n_modules'],
                'characteristic_path_length_w': metrics_w['characteristic_path_length'],
                'local_efficiency_mean_w': metrics_w['local_efficiency_mean'],
                'transitivity_w': metrics_w['transitivity'],
                'assortativity_w': metrics_w['assortativity'],
                'strength_mean': metrics_w['strength_mean'],
                # Binary metrics
                'global_efficiency_b': metrics_b['global_efficiency'],
                'clustering_mean_b': metrics_b['clustering_mean'],
                'modularity_b': metrics_b['modularity'],
                'characteristic_path_length_b': metrics_b['characteristic_path_length'],
                'degree_mean': metrics_b['degree_mean'],
            }
            
            if sw_metrics:
                sweep_row['small_worldness_sigma'] = sw_metrics['sigma']
                sweep_row['small_worldness_gamma'] = sw_metrics['gamma']
                sweep_row['small_worldness_lambda'] = sw_metrics['lambda']
            
            sweep_results.append(sweep_row)
            
            # Store primary threshold result (deterministic selection)
            if thresh == self.primary_threshold:
                primary_result = {
                    'threshold': thresh,
                    'density': density,
                    'W': W,
                    'A': A,
                    'metrics_weighted': metrics_w,
                    'metrics_binary': metrics_b,
                    'small_worldness': sw_metrics
                }
        
        # Fallback if primary_threshold wasn't in sweep (shouldn't happen)
        if primary_result is None and sweep_results:
            mid_idx = len(self.thresholds) // 2
            thresh = self.thresholds[mid_idx]
            W = threshold_matrix(conn_matrix, thresh, threshold_type=self.threshold_type, binarize=False)
            A = (W > 0).astype(float)
            np.fill_diagonal(A, 0)
            primary_result = {
                'threshold': thresh,
                'density': compute_density(A),
                'W': W,
                'A': A,
                'metrics_weighted': compute_weighted_metrics(W),
                'metrics_binary': compute_binary_metrics(A),
                'small_worldness': compute_small_worldness_binary(A, n_random=n_random) if compute_smallworld else None
            }
        
        return {
            'subject_id': subject_id,
            'session_id': session_id,
            'run_id': run_id,  # NEW: track run
            'parcellation': parcellation,
            'n_parcels': n_parcels,
            'n_timepoints': n_timepoints,
            'connectivity_matrix_raw': conn_matrix_raw,
            'connectivity_matrix': conn_matrix,
            'primary': primary_result,
            'sweep': pd.DataFrame(sweep_results)
        }
    
    def run(self, compute_smallworld=True, n_random=100, generate_figures=True,
            save_matrices=False):
        """Run the complete pipeline, processing all runs."""
        subjects = self.discover_data()
        
        all_metrics = []
        all_sweep = []
        
        for parcellation in self.parcellations:
            print(f"\n{'='*60}")
            print(f"Processing: {parcellation}")
            print(f"{'='*60}")
            
            parc_results = []
            
            for subject in tqdm(subjects, desc="Subjects"):
                subject_dir = self.xcp_dir / subject
                
                for session in find_sessions(subject_dir):
                    func_dir = subject_dir / session / 'func'
                    
                    if not func_dir.exists():
                        continue
                    
                    ptseries_files = find_ptseries_files(
                        func_dir, parcellation=parcellation, 
                        concatenated=self.use_concatenated
                    )
                    
                    if not ptseries_files:
                        continue
                    
                    # === PROCESS ALL RUNS (not just first!) ===
                    for ptseries_file in ptseries_files:
                        # Extract run ID from filename
                        run_id = self._extract_run_id(ptseries_file)
                        

                        # log file being processed
                        run_str = f"run-{run_id}" if run_id else "concatenated"
                        print(f" Processing: {subject}/{session}/{run_str} : {ptseries_file.name}")
                        try:
                            result = self.process_subject_session(
                                ptseries_file, subject, session, parcellation,
                                run_id=run_id,
                                compute_smallworld=compute_smallworld, n_random=n_random
                            )
                            
                            if result is not None:
                                parc_results.append(result)
                                
                                p = result['primary']
                                mw = p['metrics_weighted']
                                mb = p['metrics_binary']
                                sw = p['small_worldness']
                                
                                row = {
                                    'subject_id': subject,
                                    'session_id': session,
                                    'run_id': run_id,  # NEW: include run in output
                                    'parcellation': parcellation,
                                    'n_parcels': result['n_parcels'],
                                    'n_timepoints': result['n_timepoints'],
                                    'threshold': p['threshold'],
                                    'density': p['density'],
                                    'edge_type': self.edge_type,
                                    # Weighted metrics
                                    'global_efficiency_w': mw['global_efficiency'],
                                    'clustering_mean_w': mw['clustering_mean'],
                                    'modularity_w': mw['modularity'],
                                    'n_modules_w': mw['n_modules'],
                                    'characteristic_path_length_w': mw['characteristic_path_length'],
                                    'local_efficiency_mean_w': mw['local_efficiency_mean'],
                                    'transitivity_w': mw['transitivity'],
                                    'assortativity_w': mw['assortativity'],
                                    'strength_mean': mw['strength_mean'],
                                    'betweenness_mean_w': mw['betweenness_mean'],
                                    'participation_mean_w': mw['participation_mean'],
                                    # Binary metrics
                                    'global_efficiency_b': mb['global_efficiency'],
                                    'clustering_mean_b': mb['clustering_mean'],
                                    'modularity_b': mb['modularity'],
                                    'degree_mean': mb['degree_mean'],
                                    'betweenness_mean_b': mb['betweenness_mean'],
                                    'local_efficiency_mean_b': mb['local_efficiency_mean'],
                                    'assortativity_b': mb['assortativity'],
                                    'transitivity_b': mb['transitivity'],
                                }
                                
                                if sw:
                                    row['small_worldness_sigma'] = sw['sigma']
                                    row['small_worldness_gamma'] = sw['gamma']
                                    row['small_worldness_lambda'] = sw['lambda']
                                    row['C_real'] = sw['C_real']
                                    row['C_rand'] = sw['C_rand']
                                    row['L_real'] = sw['L_real']
                                    row['L_rand'] = sw['L_rand']
                                
                                all_metrics.append(row)
                                
                                sweep_df = result['sweep'].copy()
                                sweep_df['subject_id'] = subject
                                sweep_df['session_id'] = session
                                sweep_df['run_id'] = run_id  # NEW
                                sweep_df['parcellation'] = parcellation
                                all_sweep.append(sweep_df)
                                
                                if save_matrices:
                                    run_str = f"_run-{run_id}" if run_id else ""
                                    matrix_file = self.output_dir / 'connectivity_matrices' / \
                                                 f'{subject}_{session}{run_str}_{parcellation}_conn.npy'
                                    np.save(matrix_file, result['connectivity_matrix'])
                                    
                        except Exception as e:
                            print(f"  Error: {subject}/{session}/run-{run_id} - {e}")
                            import traceback
                            traceback.print_exc()
                            continue
            
            self.results[parcellation] = parc_results
            
            if generate_figures and parc_results:
                self._generate_parcellation_figures(parcellation, parc_results)
        
        self.summary_df = pd.DataFrame(all_metrics)
        self.sweep_df = pd.concat(all_sweep, ignore_index=True) if all_sweep else pd.DataFrame()
        
        # Save results
        self.summary_df.to_csv(self.output_dir / 'metrics' / 'graph_metrics_all.csv', index=False)
        
        if len(self.sweep_df) > 0:
            self.sweep_df.to_csv(self.output_dir / 'metrics' / 'threshold_sweep_all.csv', index=False)
        
        # Save per-parcellation
        for parc in self.summary_df['parcellation'].unique():
            parc_df = self.summary_df[self.summary_df['parcellation'] == parc]
            parc_df.to_csv(self.output_dir / 'metrics' / f'graph_metrics_{parc}.csv', index=False)
        
        # Count unique subjects vs total runs
        n_subjects = self.summary_df['subject_id'].nunique()
        n_sessions = self.summary_df.groupby(['subject_id', 'session_id']).ngroups
        n_runs = len(self.summary_df)
        
        print(f"\n{'='*60}")
        print("PROCESSING COMPLETE")
        print(f"{'='*60}")
        print(f"Unique subjects: {n_subjects}")
        print(f"Unique sessions: {n_sessions}")
        print(f"Total runs processed: {n_runs}")
        print(f"Edge handling: {self.edge_type}")
        print(f"Primary threshold: {self.primary_threshold}")
        print(f"Results saved to: {self.output_dir}")
        
        return self.summary_df
    
    def _generate_parcellation_figures(self, parcellation, results):
        """Generate summary figures for a parcellation."""
        print(f"  Generating figures for {parcellation}...")
        fig_dir = self.output_dir / 'figures' / parcellation
        fig_dir.mkdir(parents=True, exist_ok=True)
        
        # Group average - use best-of-N for Louvain reproducibility
        avg_matrix_raw = np.mean([r['connectivity_matrix_raw'] for r in results], axis=0)
        plot_connectivity_matrix(avg_matrix_raw, 
                                title=f'Group Average (raw r) - {parcellation} (n={len(results)} runs)',
                                save_path=fig_dir / 'group_avg_connectivity_raw.png')
        
        avg_matrix = np.mean([r['connectivity_matrix'] for r in results], axis=0)
        plot_connectivity_matrix(avg_matrix, 
                                title=f'Group Average ({self.edge_type}) - {parcellation}',
                                save_path=fig_dir / 'group_avg_connectivity.png')
        
        avg_thresh = threshold_matrix(avg_matrix, self.primary_threshold, 
                                      threshold_type=self.threshold_type, binarize=False)
        
        # Best-of-10 Louvain for group community detection (reproducibility)
        best_q = -np.inf
        best_ci = None
        for _ in range(10):
            ci_temp, q_temp = bct.community_louvain(avg_thresh)
            if q_temp > best_q:
                best_q = q_temp
                best_ci = ci_temp
        group_ci, group_q = best_ci, best_q
        
        plot_community_matrix(avg_matrix, group_ci,
                             title=f'Group Community Structure - {parcellation}\n(Q={group_q:.3f}, {len(np.unique(group_ci))} modules)',
                             save_path=fig_dir / 'group_community_structure.png')
        
        parc_df = pd.DataFrame([{
            'global_efficiency_w': r['primary']['metrics_weighted']['global_efficiency'],
            'clustering_mean_w': r['primary']['metrics_weighted']['clustering_mean'],
            'modularity_w': r['primary']['metrics_weighted']['modularity'],
            'characteristic_path_length_w': r['primary']['metrics_weighted']['characteristic_path_length'],
            'transitivity_w': r['primary']['metrics_weighted']['transitivity'],
            'small_worldness_sigma': r['primary']['small_worldness']['sigma'] if r['primary']['small_worldness'] else np.nan
        } for r in results])
        
        plot_metrics_summary(parc_df, parcellation, 
                            save_path=fig_dir / 'metrics_distribution.png')
        
        print(f"  Figures saved to: {fig_dir}")
    
    def compute_auc_metrics(self):
        """Compute Area Under Curve (AUC) for metrics across threshold sweep."""
        if self.sweep_df is None or len(self.sweep_df) == 0:
            print("No sweep data. Run pipeline first.")
            return None
        
        auc_data = []
        
        # Group by subject, session, run, and parcellation
        group_cols = ['subject_id', 'session_id', 'run_id', 'parcellation']
        
        for keys, group in self.sweep_df.groupby(group_cols):
            group = group.sort_values('threshold')
            
            row = dict(zip(group_cols, keys))
            
            metrics_for_auc = ['global_efficiency_w', 'clustering_mean_w', 'modularity_w',
                              'transitivity_w', 'small_worldness_sigma']
            
            for metric in metrics_for_auc:
                if metric in group.columns:
                    data = group[['threshold', metric]].dropna()
                    if len(data) > 1:
                        auc = np.trapz(data[metric], data['threshold'])
                        row[f'{metric}_auc'] = auc
            
            auc_data.append(row)
        
        auc_df = pd.DataFrame(auc_data)
        auc_df.to_csv(self.output_dir / 'metrics' / 'auc_metrics.csv', index=False)
        
        return auc_df
    
    def compute_session_averages(self):
        """
        Compute session-level averages by averaging across runs within each session.
        Useful for analyses that need one value per session rather than per run.
        """
        if self.summary_df is None or len(self.summary_df) == 0:
            print("No data. Run pipeline first.")
            return None
        
        # Numeric columns to average
        numeric_cols = self.summary_df.select_dtypes(include=[np.number]).columns.tolist()
        # Remove columns that shouldn't be averaged
        exclude = ['n_parcels', 'threshold']
        numeric_cols = [c for c in numeric_cols if c not in exclude]
        
        # Group by subject, session, parcellation and compute mean
        group_cols = ['subject_id', 'session_id', 'parcellation']
        session_avg = self.summary_df.groupby(group_cols)[numeric_cols].mean().reset_index()
        
        # Add metadata
        session_avg['edge_type'] = self.edge_type
        session_avg['threshold'] = self.primary_threshold
        session_avg['n_runs_averaged'] = self.summary_df.groupby(group_cols).size().values
        
        # Save
        session_avg.to_csv(self.output_dir / 'metrics' / 'graph_metrics_session_avg.csv', index=False)
        
        print(f"Session averages computed: {len(session_avg)} sessions")
        return session_avg
    
    def print_summary(self):
        """Print summary statistics."""
        if self.summary_df is None:
            print("No results. Run pipeline first.")
            return
        
        print("\n" + "="*70)
        print("GRAPH THEORY ANALYSIS SUMMARY")
        print("="*70)
        print(f"Edge handling: {self.edge_type}")
        print(f"Primary threshold: {self.primary_threshold}")
        print(f"Thresholds tested: {self.thresholds}")
        print(f"Using per-run files: {not self.use_concatenated}")
        
        # Overall counts
        n_subjects = self.summary_df['subject_id'].nunique()
        n_runs = len(self.summary_df)
        print(f"\nTotal: {n_subjects} subjects, {n_runs} runs")
        
        for parc in sorted(self.summary_df['parcellation'].unique()):
            df = self.summary_df[self.summary_df['parcellation'] == parc]
            n_subj = df['subject_id'].nunique()
            n_run = len(df)
            print(f"\n{parc} ({n_subj} subjects, {n_run} runs):")
            print("-"*50)
            
            # Show actual density achieved
            if 'density' in df.columns:
                dens = df['density'].dropna()
                print(f"  {'Actual Density':35s}: {dens.mean():.4f} ± {dens.std():.4f}")
            
            metrics = [
                ('Global Efficiency (weighted)', 'global_efficiency_w'),
                ('Clustering (weighted)', 'clustering_mean_w'),
                ('Modularity (weighted)', 'modularity_w'),
                ('Path Length (weighted)', 'characteristic_path_length_w'),
                ('Transitivity (weighted)', 'transitivity_w'),
                ('Small-worldness (σ)', 'small_worldness_sigma'),
                ('Normalized Clustering (γ)', 'small_worldness_gamma'),
                ('Normalized Path Length (λ)', 'small_worldness_lambda'),
            ]
            
            for name, col in metrics:
                if col in df.columns:
                    data = df[col].replace([np.inf, -np.inf], np.nan).dropna()
                    if len(data) > 0:
                        print(f"  {name:35s}: {data.mean():.4f} ± {data.std():.4f}")
        
        print("\n" + "="*70)

In [ ]:

# %% Cell 8: Run the Pipeline
# Initialize pipeline with CORRECTED settings
pipeline = XCPGraphTheoryPipeline(
    xcp_dir=XCP_OUTPUT_DIR,
    output_dir=OUTPUT_DIR,
    parcellations=PARCELLATIONS,
    use_concatenated=USE_CONCATENATED,
    thresholds=THRESHOLDS,  # Threshold sweep
    threshold_type=THRESHOLD_TYPE,
    edge_type=EDGE_TYPE  # 'positive_only' recommended
)

# Run analysis
summary_df = pipeline.run(
    compute_smallworld=COMPUTE_SMALLWORLD,
    n_random=N_RANDOM,
    generate_figures=True,
    save_matrices=False
)

# Print summary
pipeline.print_summary()



Discovering XCP-D data...
Using per-run files: True
Found 1 subjects
Available parcellations: ['4S1056Parcels', '4S156Parcels', '4S256Parcels', '4S356Parcels', '4S456Parcels', '4S556Parcels', '4S656Parcels', '4S756Parcels', '4S856Parcels', '4S956Parcels', 'Glasser', 'Gordon', 'HCP', 'MIDB', 'MyersLabonte', 'Tian']
Will process: ['4S456Parcels', 'Glasser', 'Gordon']
Edge handling: positive_only
Thresholds: [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
Primary threshold: 0.15

Processing: 4S456Parcels


Subjects:   0%|                                                                                                                        | 0/1 [00:00<?, ?it/s]

 Processing: sub-CU1CU10001/ses-02/run-01 : sub-CU1CU10001_ses-02_task-rest_run-01_space-fsLR_seg-4S456Parcels_den-91k_stat-mean_timeseries.ptseries.nii


Subjects:   0%|                                                                                                                        | 0/1 [00:11<?, ?it/s]


KeyboardInterrupt: 

: 

In [ ]:
# %% Cell 9: Compute AUC Metrics (threshold-independent)
auc_df = pipeline.compute_auc_metrics()
if auc_df is not None:
    print("\nAUC Metrics (threshold-independent):")
    print(auc_df.head())



In [ ]:
# %% Cell 10: View Results
print("\nResults Preview:")
print(summary_df.head())

print("\nKey columns:")
print(summary_df.columns.tolist())



In [ ]:
# %% Cell 11: Compare Parcellations
if len(summary_df['parcellation'].unique()) > 1:
    fig_dir = pipeline.output_dir / 'figures' / 'parcellation_comparison'
    fig_dir.mkdir(parents=True, exist_ok=True)
    
    metrics = ['global_efficiency_w', 'clustering_mean_w', 'modularity_w',
              'small_worldness_sigma', 'transitivity_w']
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.flatten()
    
    for i, metric in enumerate(metrics):
        if metric not in summary_df.columns:
            continue
        
        ax = axes[i]
        parcellations = sorted(summary_df['parcellation'].unique())
        
        data = [summary_df[summary_df['parcellation'] == p][metric].replace([np.inf, -np.inf], np.nan).dropna() 
               for p in parcellations]
        
        bp = ax.boxplot(data, labels=parcellations, patch_artist=True)
        
        colors = plt.cm.tab10(np.linspace(0, 1, len(parcellations)))
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax.set_ylabel(metric.replace('_', ' ').title())
        ax.set_title(metric.replace('_', ' ').title(), fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
    
    axes[-1].set_visible(False)
    
    plt.suptitle(f'Graph Metrics Across Parcellations\n(Edge type: {EDGE_TYPE})', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(fig_dir / 'parcellation_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()



In [ ]:
# %% Cell 12: Export for Statistical Analysis
print("\nExporting results...")

# Per-run results
for parc in summary_df['parcellation'].unique():
    parc_df = summary_df[summary_df['parcellation'] == parc].copy()
    parc_df.to_csv(f'{OUTPUT_DIR}/metrics/export_{parc}_per_run.csv', index=False)
    n_subj = parc_df['subject_id'].nunique()
    n_runs = len(parc_df)
    print(f"Exported: export_{parc}_per_run.csv ({n_subj} subjects, {n_runs} runs)")

# Session-averaged results (average across runs within session)
print("\nComputing session averages...")
session_avg_df = pipeline.compute_session_averages()
if session_avg_df is not None:
    for parc in session_avg_df['parcellation'].unique():
        parc_df = session_avg_df[session_avg_df['parcellation'] == parc].copy()
        parc_df.to_csv(f'{OUTPUT_DIR}/metrics/export_{parc}_session_avg.csv', index=False)
        print(f"Exported: export_{parc}_session_avg.csv ({len(parc_df)} sessions)")

print(f"\nAll results saved to: {OUTPUT_DIR}")
print("\nOutput files:")
print("  - graph_metrics_all.csv: All runs, all parcellations")
print("  - graph_metrics_<parc>.csv: All runs for specific parcellation")
print("  - graph_metrics_session_avg.csv: Averaged across runs per session")
print("  - export_<parc>_per_run.csv: For per-run statistical analysis")
print("  - export_<parc>_session_avg.csv: For session-level statistical analysis")